# 0. Environment Setup

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## 0.1 Clone Repository & Install Dependencies

In [2]:
REPO_URL = "https://github.com/11erlangga/legal-rag-slm.git"
REPO_DIR = "/kaggle/working/repo"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 98 (delta 56), reused 75 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (98/98), 404.40 KiB | 7.63 MiB/s, done.
Resolving deltas: 100% (56/56), done.


In [3]:
!pip install -q "transformers==4.57.3" "accelerate==1.12.0" "bitsandbytes==0.49.1"
!pip install -q "langchain==1.2.3" "langchain-community==0.4.1" "langchain-core==1.2.6"
!pip install -q langchain-classic langchain-text-splitters langchain-huggingface langchain-chroma
!pip install -q "chromadb==1.4.1" "huggingface-hub==0.36.0"
!pip install -q rank-bm25 sentence-transformers pymupdf ddgs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 489.1/489.1 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curren

## 0.2 Import Libraries & Secrets

In [4]:
from IPython.display import display, Markdown

import sys
sys.path.append(REPO_DIR)

import re
import torch
from typing import Dict, Optional

from src.rag.pipeline import (
    build_retrievers, build_generator, build_pipeline,
    RAGPipeline, sanity_check_retrieval,
)
from src.rag.hyde import build_hyde_pipeline, generate_hypothetical_answers, hyde_retrieve
from src.rag.retrievers import print_retrieved_docs, rerank_with_scores
from src.rag.web_fallback import retrieve_with_fallback
from src.rag.pipeline import _format_source_line

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

In [5]:
if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU Detected: {gpu_stats.name}. Max Memory = {max_memory} GB.")
else:
    print("WARNING: GPU Not Detected.")

GPU Detected: Tesla T4. Max Memory = 14.562 GB.


# 1. Config

In [6]:
PDF_DIR = "/kaggle/input/datasets/erlanggasriheryanto/uu-legal-docs"
HF_REPO_ID = "11erlangga/grpo-qwen25-3b"

ENSEMBLE_WEIGHTS = (0.75, 0.25) # BM25, semantic
RERANKER_TOP_N = 3
HYDE_N = 2
FALLBACK_THRESHOLD = 0.0 # BELUM dikalibrasi

# 2. Build Pipeline Final (Hybrid + HyDE + Fallback)

In [7]:
pipeline = build_pipeline(
    pdf_dir=PDF_DIR,
    hf_repo_id=HF_REPO_ID,
    ensemble_weights=ENSEMBLE_WEIGHTS,
    reranker_top_n=RERANKER_TOP_N,
    hf_token=HF_TOKEN,
    use_hyde=True,
    hyde_n=HYDE_N,
    use_fallback=True,
    fallback_threshold=FALLBACK_THRESHOLD,
)

Parent chunk size: 2000, overlap: 200
Child chunk size: 400, overlap: 50


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/240 [00:00<?, ?B/s]

Device set to use cuda:0
Device set to use cuda:0


# 3. Test Case

In [8]:
TEST_CASE_QUERY = ("Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?")

result = pipeline.generate(TEST_CASE_QUERY)

display(Markdown(f"**Output Mentah:**\n\n```\n{result['answer']}\n```"))

**Output Mentah:**

```
Berdasarkan konteks dokumen yang diberikan, Anda berhak mendapatkan upah lembur karena Anda melakukan kerja lembur selama 3 jam. Namun, hal ini tergantung pada aturan dan ketentuan yang ditetapkan oleh perusahaan Anda mengenai waktu kerja lembur. Sebaiknya Anda berkonsultasi dengan manajemen atau karyawan HR untuk memastikan bahwa Anda berhak mendapatkan upah lembur sesuai dengan aturan yang berlaku.
```

In [9]:
print(f"used_fallback: {result['used_fallback']} | top_score: {result['top_score']}")
print()
display(Markdown("**SUMBER:**"))
for i, doc in enumerate(result["sources"], start=1):
    print(_format_source_line(doc, i))

used_fallback: False | top_score: 0.7675754427909851



**SUMBER:**

1. PP No. 35 Tahun 2021, PP Nomor 35 Tahun 2021.pdf, halaman 18
2. PP No. 35 Tahun 2021, PP Nomor 35 Tahun 2021.pdf, halaman 17 — Pasal: 23, 2l, 30, 31
3. PP No. 35 Tahun 2021, PP Nomor 35 Tahun 2021.pdf, halaman 16 — Pasal: 26, 27, 28, 2l


# 4. Verifikasi Terstruktur

In [10]:
PP35_PATTERN = re.compile(
    r"(pp|peraturan\s+pemerintah).{0,20}35.{0,20}2021",
    re.IGNORECASE | re.DOTALL,
)


def verify_test_case_output(answer: str) -> dict:
    """Memverifikasi output test case wajib terhadap 3 kriteria brief.

    Args:
        answer: Teks jawaban mentah dari `pipeline.generate()["answer"]`.

    Returns:
        Dict berisi hasil pengecekan tiap kriteria plus verdict akhir
        `"passed"` (True hanya jika ketiga kriteria terpenuhi).
    """
    has_open = "<think>" in answer
    has_close = "</think>" in answer
    tag_order_correct = (
        has_open and has_close and answer.index("<think>") < answer.index("</think>")
    )

    if has_close:
        after_think = answer.split("</think>", 1)[1].strip()
        has_final_answer = len(after_think) > 0
    else:
        after_think = ""
        has_final_answer = False

    mentions_pp35 = bool(PP35_PATTERN.search(answer))

    passed = tag_order_correct and has_final_answer and mentions_pp35

    return {
        "tag_order_correct": tag_order_correct,
        "has_final_answer": has_final_answer,
        "mentions_pp_35_2021": mentions_pp35,
        "final_answer_preview": after_think[:200],
        "passed": passed,
    }


verification = verify_test_case_output(result["answer"])
for key, value in verification.items():
    print(f"{key}: {value}")


tag_order_correct: False
has_final_answer: False
mentions_pp_35_2021: False
final_answer_preview: 
passed: False


# 5. Catatan untuk README

In [ ]:
with open("/kaggle/working/test_case_result.md", "w", encoding="utf-8") as f:
    f.write("# Test Case Wajib -- Hasil Run NB05\n\n")
    f.write(f"**Query:** {TEST_CASE_QUERY}\n\n")
    f.write(f"**used_fallback:** {result['used_fallback']}\n\n")
    f.write(f"**top_score:** {result['top_score']}\n\n")
    f.write("## Output Mentah\n\n```\n" + result["answer"] + "\n```\n\n")
    f.write("## Verifikasi\n\n")
    for key, value in verification.items():
        f.write(f"- **{key}**: {value}\n")

print("Tersimpan ke /kaggle/working/test_case_result.md")

Tersimpan ke /kaggle/working/test_case_result.md
